<a href="https://colab.research.google.com/github/beksultann-GQ/Bexs_work/blob/main/03_lime_pdp.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Зертханалық жұмыс 3: LIME және Partial Dependence Plots арқылы модельді интерпретациялау


Осы зертханалық жұмыс модельдерді түсіндіру бойынша үш серияның қорытынды бөлігі болып саналады. Бірінші жұмыста біз глобалды feature importance әдістерін қарадық, екіншісінде SHAP арқылы Shapley мәндерімен таныстық. Бұл жолы екі толықтырушы әдісті қарастырамыз: LIME (Local Interpretable Model-agnostic Explanations) және Partial Dependence Plots (PDP) мен Individual Conditional Expectation (ICE) графиктері.

## Оқу мақсаттары

Жұмыс соңында студент LIME кітапханасын пайдаланып, кез келген қара жәшік (black-box) модель үшін жеке болжамды локалды түрде түсіндіре алады. PDP және ICE графиктерін салып, белгінің болжамға орташа және жеке-жеке әсерін ажырата алады. Сонымен қатар LIME, SHAP және PDP әдістерін бір-бірімен салыстырып, қай жағдайда қайсысын пайдаланған дұрыс екенін нақты дәйектей алатын болады.

## Жұмыс құрылымы

1-бөлім. Ортаны дайындау және LIME орнату.
2-бөлім. California Housing регрессиялық деректер жинағы.
3-бөлім. Random Forest регрессорын оқыту.
4-бөлім. LIME арқылы жеке болжамды түсіндіру.
5-бөлім. Partial Dependence Plots (PDP).
6-бөлім. Individual Conditional Expectation (ICE) графиктері.
7-бөлім. Әдістерді салыстыру.
8-бөлім. Рефлексия сұрақтары.

## 1-бөлім. Ортаны дайындау

LIME кітапханасын pip арқылы орнатамыз. PDP және ICE үшін scikit-learn ішіндегі PartialDependenceDisplay класы жеткілікті, қосымша орнату қажет емес.

In [ ]:
# LIME орнату
# !pip install lime --quiet

In [ ]:
# Негізгі кітапханалар
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.inspection import PartialDependenceDisplay
from sklearn.metrics import mean_squared_error, r2_score

# LIME
import lime
import lime.lime_tabular

# Кездейсоқтықты бекіту
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

plt.rcParams["figure.figsize"] = (8, 5)

## 2-бөлім. Деректерді жүктеу

Бұл жұмыста California Housing деректер жинағын пайдаланамыз. Мақсат — Калифорния штатының әр аймағы үшін үй құнының медианасын (жүз мыңдаған долларда) болжау. Бұл регрессия есебі болғандықтан, түсіндіру әдістері үздіксіз шығысқа бейімделген нұсқада қолданылады.

In [ ]:
# Деректерді жүктеу
housing = fetch_california_housing(as_frame=True)
df = housing.frame.copy()

# Кестенің басы
df.head()

### ### TODO ###

Деректер жинағының сипаттамасын шығарыңыз және сандық статистикасын көріңіз.

In [ ]:
### ### TODO ###
# 1. Жалпы сипаттамасын шығару
# print(housing.____)

# 2. Сандық статистикасын көрсету
# df.____()

In [ ]:
# Белгілер мен мақсатты айнымалыны бөлеміз
X = df.drop(columns=["MedHouseVal"])
y = df["MedHouseVal"]

# Оқу және тест жиынтықтарына бөлу
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=RANDOM_STATE
)

print("Оқу:", X_train.shape, "Тест:", X_test.shape)

## 3-бөлім. Random Forest регрессорын оқыту

LIME модельге тәуелсіз әдіс болғандықтан, қандай да бір қара жәшік модель қажет. Біз Random Forest регрессорын оқытамыз, бірақ дәл сол LIME кодын нейрондық желіге де, XGBoost-қа да қолдана алар едік.

### ### TODO ###

Random Forest регрессорын құрып, оқытыңыз. Ағаштар саны 200, максималды тереңдігі 12, n_jobs=-1 болсын.

In [ ]:
### ### TODO ###
# rf_reg = RandomForestRegressor(
#     n_estimators=____,
#     max_depth=____,
#     random_state=RANDOM_STATE,
#     n_jobs=____
# )

# rf_reg.fit(____, ____)

# Тест жиынтығында бағалау
# y_pred = rf_reg.predict(____)
# rmse = np.sqrt(mean_squared_error(y_test, y_pred))
# r2 = r2_score(y_test, y_pred)
# print(f"RMSE: {rmse:.4f}")
# print(f"R^2: {r2:.4f}")

## 4-бөлім. LIME арқылы жеке болжамды түсіндіру

LIME идеясы қарапайым. Қара жәшік модельдің шешімін жеке нүкте маңайында жуықтап көрсету үшін ол сол нүктенің айналасында жасанды үлгілер жасайды, оларға модель болжамын алады, содан кейін салмақталған сызықтық регрессия оқытады. Осы жергілікті сызықтық модель шешімнің қалай жасалғанын түсіндіреді.

In [ ]:
# LIME explainer инициализациясы
lime_explainer = lime.lime_tabular.LimeTabularExplainer(
    training_data=X_train.values,
    feature_names=X_train.columns.tolist(),
    mode="regression",
    random_state=RANDOM_STATE
)

### ### TODO ###

Тест жиынтығынан бір мысалды таңдап, LIME түсіндірмесін алыңыз. num_features=6 (яғни ең маңызды 6 белгіні көрсету).

In [ ]:
### ### TODO ###
# Талданатын мысалдың индексі
# idx = 10

# Модель болжамы
# actual = y_test.iloc[idx]
# predicted = rf_reg.predict(X_test.iloc[[idx]])[0]
# print(f"Нақты мән: {actual:.3f}")
# print(f"Модель болжамы: {predicted:.3f}")

# LIME түсіндірмесін алу
# explanation = lime_explainer.explain_instance(
#     data_row=X_test.iloc[____].values,
#     predict_fn=rf_reg.____,     # регрессия үшін predict функциясы
#     num_features=____
# )

# Нәтижелерді кесте түрінде көру
# pd.DataFrame(explanation.as_list(), columns=["Белгі шарты", "Салмақ"])

LIME нәтижесін визуализациялап көрейік. as_pyplot_figure() әдісі әр белгінің үлесін бағаналы диаграмма түрінде көрсетеді. Оң мәндер болжамды жоғары қарай, теріс мәндер төмен қарай жылжытады.

In [ ]:
# LIME графигін салу
fig = explanation.as_pyplot_figure()
plt.title(f"LIME түсіндірмесі, мысал №{idx}")
plt.tight_layout()
plt.show()

**Талдау сұрағы 1.** Осы нақты мысалда модель болжамын жоғары қарай жылжытқан ең күшті екі белгіні атаңыз. Олардың мәні нақты қандай еді және неге сіздіңше олар осындай әсер берді?

### ### TODO ###

Тағы бір мысалды алыңыз (мысалы, idx=100) және түсіндірмені қайта жасаңыз. Екі мысал үшін таңдалған белгілер бірдей ме? Егер олай болмаса, бұл нені білдіреді?

In [ ]:
### ### TODO ###
# idx2 = ____
# actual2 = y_test.iloc[idx2]
# predicted2 = rf_reg.predict(X_test.iloc[[idx2]])[0]
# print(f"Нақты мән: {actual2:.3f}, болжам: {predicted2:.3f}")

# exp2 = lime_explainer.explain_instance(
#     data_row=____,
#     predict_fn=____,
#     num_features=6
# )

# fig2 = exp2.____()
# plt.title(f"LIME түсіндірмесі, мысал №{idx2}")
# plt.tight_layout()
# plt.show()

**Талдау сұрағы 2.** LIME-ның бір ерекшелігі — жергілікті түсіндірмелер бір-бірінен ерекшеленуі мүмкін, себебі әр нүктенің төңірегінде модель әртүрлі мінез көрсетеді. Бұл қасиет практикада артықшылық па, әлде кемшілік пе? Бір абзацпен пайымдаңыз.

## 5-бөлім. Partial Dependence Plots (PDP)

PDP белгінің болжамға тигізетін орташа шекті әсерін көрсетеді. Идеясы: белгінің мәнін бекітілген бір санға өзгертіп, қалған белгілерді өз орнында қалдырамыз, содан кейін модельдің орташа болжамын есептейміз. Осы процедураны белгінің әр мәні үшін қайталап, қисық сызамыз.

In [ ]:
# Бір белгі бойынша PDP
fig, ax = plt.subplots(figsize=(8, 5))
PartialDependenceDisplay.from_estimator(
    rf_reg,
    X_train,
    features=["MedInc"],
    grid_resolution=50,
    ax=ax
)
plt.suptitle("PDP: MedInc белгісі")
plt.tight_layout()
plt.show()

### ### TODO ###

Бірден бірнеше белгі бойынша PDP салыңыз. Features тізіміне MedInc, HouseAge, AveRooms және AveOccup енгізіңіз.

In [ ]:
### ### TODO ###
# fig, ax = plt.subplots(figsize=(12, 8))
# PartialDependenceDisplay.from_estimator(
#     rf_reg,
#     X_train,
#     features=[____, ____, ____, ____],
#     grid_resolution=50,
#     ax=ax
# )
# plt.tight_layout()
# plt.show()

PDP екі белгі үшін де салынуы мүмкін. Бұл жағдайда контурлы график шығады, ол екі белгінің өзара әсерін көрсетеді.

In [ ]:
# Екі белгі бойынша PDP
fig, ax = plt.subplots(figsize=(8, 6))
PartialDependenceDisplay.from_estimator(
    rf_reg,
    X_train,
    features=[("MedInc", "AveOccup")],
    grid_resolution=25,
    ax=ax
)
plt.suptitle("PDP: MedInc пен AveOccup өзара әсері")
plt.tight_layout()
plt.show()

**Талдау сұрағы 3.** MedInc (табыс медианасы) белгісінің PDP қисығы қандай пішінде? Ол сызықты ма, әлде белгілі бір нүктеде тегістеле ме? Мұны экономикалық тұрғыдан қалай түсіндіруге болады?

## 6-бөлім. Individual Conditional Expectation (ICE)

PDP тек орташа әсерді көрсетеді. Кейде әр жеке бақылау үшін белгінің әсері әртүрлі болуы мүмкін, ал орташа шама бұл айырмашылықты жасырады. ICE графигі әр бақылау үшін жеке қисық сызып, PDP-ны толықтырады.

### ### TODO ###

MedInc белгісі үшін ICE және PDP қисықтарын бір графикте көрсетіңіз. kind="both" параметрі PDP үстіне ICE қисықтарын салады.

In [ ]:
### ### TODO ###
# fig, ax = plt.subplots(figsize=(8, 5))
# PartialDependenceDisplay.from_estimator(
#     rf_reg,
#     X_train.sample(500, random_state=RANDOM_STATE),   # есептеуді жеңілдету
#     features=[____],
#     kind="____",
#     grid_resolution=50,
#     ax=ax
# )
# plt.suptitle("PDP + ICE: MedInc")
# plt.tight_layout()
# plt.show()

**Талдау сұрағы 4.** ICE қисықтары бір-біріне параллель ме, әлде айқын әр түрлі бағыттарға айырылады ма? Егер айырылатын болса, бұл модельде MedInc мен басқа белгі арасында өзара әсер (interaction) бар екенін білдіреді. Не байқадыңыз?

## 7-бөлім. Әдістерді салыстыру

Осы уақытқа дейін біз үш негізгі әдіспен таныстық: feature importance (глобалды), SHAP (глобалды + локалды), LIME (локалды), PDP және ICE (глобалды, бір белгі). Төмендегі кестеде олардың сипаттамалары қысқаша салыстырылған.

In [ ]:
# Әдістерді салыстыру кестесі
comparison_df = pd.DataFrame({
    "Әдіс": ["Feature Importance", "Permutation Imp.", "SHAP", "LIME", "PDP", "ICE"],
    "Ауқымы": ["Глобалды", "Глобалды", "Екеуі де", "Локалды", "Глобалды", "Локалды"],
    "Модельге тәуелсіз бе": ["Жоқ", "Иә", "Ішінара", "Иә", "Иә", "Иә"],
    "Есептеу шығыны": ["Төмен", "Орташа", "Орташа/Жоғары", "Орташа", "Орташа", "Жоғары"],
    "Себеп-салдарлықты бере ме": ["Жоқ", "Жоқ", "Жоқ", "Жоқ", "Жоқ", "Жоқ"]
})
comparison_df

### ### TODO ###

Соңғы тапсырма ретінде бір ғана мысалды алып, оған үш әдіс те қолданыңыз: LIME түсіндірмесі, SHAP waterfall (қажет болса алдыңғы зертханадан SHAP-ты орнатып), және сол мысалдағы модель болжамын. Нәтижелерді қатар қойып, ұқсастық пен айырмашылықты сипаттаңыз.

In [ ]:
### ### TODO ###
# 1. Мысалды таңдау
# idx_final = ____

# 2. LIME түсіндірмесі
# exp_final = lime_explainer.explain_instance(
#     data_row=____,
#     predict_fn=rf_reg.predict,
#     num_features=____
# )
# print("LIME белгілері:")
# for f, w in exp_final.as_list():
#     print(f"  {f}: {w:+.4f}")

# 3. Нақты мән мен болжам
# print(f"Нақты мән: {y_test.iloc[idx_final]:.3f}")
# print(f"Модель болжамы: {rf_reg.predict(X_test.iloc[[idx_final]])[0]:.3f}")

**Талдау сұрағы 5.** LIME мен алдыңғы зертханадағы SHAP әдісі бір жеке болжам үшін берген ең маңызды белгілердің тізімін салыстырыңыз (концептуалды түрде, дәл сол мысал болмаса да). Олардың арасында айырмашылық болу нені білдіреді және бұл жағдайда практик қандай қорытынды жасауы керек?

## Қорытынды

Осы жұмыста сіз локалды түсіндірудің тағы бір негізгі әдісі — LIME-ды меңгердіңіз және глобалды белгі-болжам байланысын көрсететін PDP мен ICE графиктерімен таныстыңыз. LIME қара жәшік модельдер үшін жедел болатын, түсінікті, бірақ тұрақсыз (жасанды үлгілерге тәуелді) түсіндірме береді. PDP модельдің жалпы логикасын көрсетеді, ал ICE жеке бақылаулардың бірегей мінез-құлқын ашады. Үш зертханалық жұмыстың нәтижесі бойынша сіз модель интерпретациясының толық құралдар жиынтығын иеленесіз: глобалды деңгейде feature importance және PDP, локалды деңгейде SHAP және LIME. Практикада бұл әдістерді бір-бірімен толықтырып қолданған дұрыс, себебі әрбір әдістің өз артықшылықтары мен шектеулері бар.

## Рефлексия сұрақтары

1. LIME шығаратын түсіндірмелер бір мысал үшін әр іске қосу кезінде сәл-пәл өзгеруі мүмкін (себебі жасанды үлгілер кездейсоқ құрылады). Мұны қалай жеңілдетуге болады және өнеркәсіптік қолданбада мұны қалай ескеру керек?

2. PDP-нің бір негізгі болжамы — белгілердің тәуелсіздігі. Егер белгілер өзара күшті корреляцияланған болса, PDP жаңылыстыруы мүмкін. Неге? Осы жағдайда қандай балама әдіс ұсынар едіңіз?

3. Осы зертхана мен алдыңғыларда жасалған жұмыс негізінде, банктік несие моделін өнеркәсіптік қолданысқа шығармас бұрын міндетті түрде жасалуы тиіс интерпретация қадамдарының қысқаша тізімін (checklist) жасаңыз.